# Course 04: Create ML Models with BigQuery ML

**CareerAlign GCP Machine Learning Engineer Certification**

This notebook is the companion to the Course 04 study guide. It provides hands-on practice with
BigQuery ML: creating models with SQL, evaluating performance, generating predictions, and
exploring feature engineering and explainability.

**Exam Sections:** Section 1 (Low-code ML), Section 3 (Scaling Prototypes)

---

### Prerequisites
- A GCP project with BigQuery API enabled
- BigQuery dataset created (we will create one below)
- `google-cloud-bigquery` Python package installed

In [ ]:
# Install required packages
!pip install google-cloud-bigquery db-dtypes pyarrow pandas matplotlib --quiet

print("Packages installed successfully.")

## 1. Connect to BigQuery

Initialize the BigQuery client. In GCP notebook environments (Colab Enterprise,
Vertex AI Workbench), authentication is handled automatically.

In [ ]:
from google.cloud import bigquery
import pandas as pd

# Initialize BigQuery client
client = bigquery.Client()
PROJECT_ID = client.project

print(f"Connected to project: {PROJECT_ID}")

# Create a dataset for our models (if it doesn't exist)
DATASET_ID = f"{PROJECT_ID}.bqml_tutorial"

dataset = bigquery.Dataset(DATASET_ID)
dataset.location = "US"
try:
    dataset = client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset {DATASET_ID} is ready.")
except Exception as e:
    print(f"Dataset creation: {e}")

## 2. Explore the Sample Dataset

We will use the public **penguins** dataset from `bigquery-public-data.ml_datasets.penguins`.
This is a classic ML dataset with species classification and body measurements.

In [ ]:
# Explore the penguins dataset
query = """
SELECT *
FROM `bigquery-public-data.ml_datasets.penguins`
WHERE body_mass_g IS NOT NULL
LIMIT 1000
"""

df = client.query(query).to_dataframe()
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSpecies distribution:")
print(df["species"].value_counts())
print(f"\nSample rows:")
df.head(10)

In [ ]:
# Summary statistics
print("Numerical summary:")
df.describe()

## 3. CREATE MODEL for Classification (Logistic Regression)

Train a logistic regression model to predict penguin species from body measurements.
This is the simplest supervised model type in BQML.

In [ ]:
# Create a logistic regression model for species classification
create_model_query = f"""
CREATE OR REPLACE MODEL `{DATASET_ID}.penguin_logistic`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['species'],
  auto_class_weights = TRUE,
  max_iterations = 20,
  data_split_method = 'AUTO_SPLIT'
) AS
SELECT
  species,
  island,
  culmen_length_mm,
  culmen_depth_mm,
  flipper_length_mm,
  body_mass_g,
  sex
FROM
  `bigquery-public-data.ml_datasets.penguins`
WHERE
  body_mass_g IS NOT NULL
  AND sex IS NOT NULL
"""

print("Training logistic regression model...")
print("(This may take 1-3 minutes)")

job = client.query(create_model_query)
job.result()  # Wait for completion

print(f"\nModel created: {DATASET_ID}.penguin_logistic")
print(f"Training time: {job.ended - job.started}")

## 4. ML.EVALUATE — Interpret Results

Evaluate the trained model's performance on the automatically split evaluation data.

In [ ]:
# Evaluate the logistic regression model
eval_query = f"""
SELECT *
FROM ML.EVALUATE(
  MODEL `{DATASET_ID}.penguin_logistic`
)
"""

eval_df = client.query(eval_query).to_dataframe()
print("=" * 60)
print("LOGISTIC REGRESSION - EVALUATION METRICS")
print("=" * 60)

for col in eval_df.columns:
    val = eval_df[col].iloc[0]
    if isinstance(val, float):
        print(f"  {col:30s} {val:.4f}")
    else:
        print(f"  {col:30s} {val}")

In [ ]:
# View the training curve
training_info_query = f"""
SELECT *
FROM ML.TRAINING_INFO(
  MODEL `{DATASET_ID}.penguin_logistic`
)
ORDER BY iteration
"""

training_df = client.query(training_info_query).to_dataframe()
print("Training history:")
training_df

In [ ]:
# View confusion matrix
confusion_query = f"""
SELECT *
FROM ML.CONFUSION_MATRIX(
  MODEL `{DATASET_ID}.penguin_logistic`
)
"""

confusion_df = client.query(confusion_query).to_dataframe()
print("Confusion Matrix:")
confusion_df

## 5. ML.PREDICT — Generate Predictions

Use the trained model to make predictions on data.

In [ ]:
# Generate predictions
predict_query = f"""
SELECT
  predicted_species,
  predicted_species_probs,
  species AS actual_species,
  island,
  culmen_length_mm,
  body_mass_g
FROM ML.PREDICT(
  MODEL `{DATASET_ID}.penguin_logistic`,
  (
    SELECT *
    FROM `bigquery-public-data.ml_datasets.penguins`
    WHERE body_mass_g IS NOT NULL
      AND sex IS NOT NULL
    LIMIT 20
  )
)
"""

predictions_df = client.query(predict_query).to_dataframe()
print("Predictions (first 20 rows):")
print(f"\nAccuracy on sample: {(predictions_df['predicted_species'] == predictions_df['actual_species']).mean():.2%}")
predictions_df[["predicted_species", "actual_species", "island", "culmen_length_mm", "body_mass_g"]]

## 6. CREATE MODEL for Time Series (ARIMA_PLUS)

Train an ARIMA_PLUS model on a public time series dataset. We will use the
Iowa liquor sales dataset for forecasting.

In [ ]:
# First, explore the time series data
ts_explore_query = """
SELECT
  DATE(date) AS sale_date,
  SUM(sale_dollars) AS total_sales
FROM
  `bigquery-public-data.iowa_liquor_sales.sales`
WHERE
  date >= '2020-01-01'
  AND date < '2023-01-01'
GROUP BY sale_date
ORDER BY sale_date
LIMIT 20
"""

ts_df = client.query(ts_explore_query).to_dataframe()
print(f"Time series sample ({len(ts_df)} rows):")
ts_df

In [ ]:
# Create an ARIMA_PLUS model for sales forecasting
arima_query = f"""
CREATE OR REPLACE MODEL `{DATASET_ID}.sales_arima`
OPTIONS (
  model_type = 'ARIMA_PLUS',
  time_series_timestamp_col = 'sale_date',
  time_series_data_col = 'total_sales',
  auto_arima = TRUE,
  holiday_region = 'US',
  data_frequency = 'DAILY'
) AS
SELECT
  DATE(date) AS sale_date,
  SUM(sale_dollars) AS total_sales
FROM
  `bigquery-public-data.iowa_liquor_sales.sales`
WHERE
  date >= '2020-01-01'
  AND date < '2023-01-01'
GROUP BY sale_date
"""

print("Training ARIMA_PLUS model...")
print("(This may take 3-5 minutes)")

job = client.query(arima_query)
job.result()

print(f"\nModel created: {DATASET_ID}.sales_arima")
print(f"Training time: {job.ended - job.started}")

In [ ]:
# Evaluate ARIMA model
arima_eval_query = f"""
SELECT *
FROM ML.EVALUATE(
  MODEL `{DATASET_ID}.sales_arima`
)
"""

arima_eval_df = client.query(arima_eval_query).to_dataframe()
print("ARIMA_PLUS Evaluation:")
arima_eval_df

In [ ]:
# Generate 30-day forecast
forecast_query = f"""
SELECT *
FROM ML.FORECAST(
  MODEL `{DATASET_ID}.sales_arima`,
  STRUCT(30 AS horizon, 0.95 AS confidence_level)
)
"""

forecast_df = client.query(forecast_query).to_dataframe()
print(f"30-day forecast ({len(forecast_df)} rows):")
forecast_df.head(10)

## 7. ML.EXPLAIN_PREDICT for Feature Importance

Use `ML.EXPLAIN_PREDICT` to understand which features drive each prediction.
This uses Shapley values for feature attribution.

In [ ]:
# Get predictions with feature attributions
explain_query = f"""
SELECT *
FROM ML.EXPLAIN_PREDICT(
  MODEL `{DATASET_ID}.penguin_logistic`,
  (
    SELECT *
    FROM `bigquery-public-data.ml_datasets.penguins`
    WHERE body_mass_g IS NOT NULL
      AND sex IS NOT NULL
    LIMIT 5
  ),
  STRUCT(3 AS top_k_features)
)
"""

explain_df = client.query(explain_query).to_dataframe()
print("Predictions with Feature Attributions:")
print(f"Columns: {list(explain_df.columns)}")
explain_df

In [ ]:
# Global feature importance
global_explain_query = f"""
SELECT *
FROM ML.GLOBAL_EXPLAIN(
  MODEL `{DATASET_ID}.penguin_logistic`
)
ORDER BY attribution DESC
"""

try:
    global_df = client.query(global_explain_query).to_dataframe()
    print("Global Feature Importance:")
    print(global_df)
except Exception as e:
    print(f"Note: ML.GLOBAL_EXPLAIN may not be available for all model types.")
    print(f"Error: {e}")

## 8. Feature Engineering with TRANSFORM Clause

The `TRANSFORM` clause embeds preprocessing into the model, ensuring consistent
transformations during both training and prediction.

In [ ]:
# Create a model with TRANSFORM clause for feature engineering
transform_query = f"""
CREATE OR REPLACE MODEL `{DATASET_ID}.penguin_engineered`
TRANSFORM (
  -- Pass through the label
  species,

  -- Bucketize body mass into categories
  ML.BUCKETIZE(body_mass_g, [3000, 4000, 5000]) AS mass_category,

  -- Quantile bucketize culmen length
  ML.QUANTILE_BUCKETIZE(culmen_length_mm, 5) AS culmen_length_quintile,

  -- Feature cross: island x sex interaction
  ML.FEATURE_CROSS(STRUCT(island, sex)) AS island_sex_cross,

  -- Pass through numeric features
  culmen_depth_mm,
  flipper_length_mm,

  -- Derived feature: bill ratio
  culmen_length_mm / NULLIF(culmen_depth_mm, 0) AS bill_ratio
)
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['species'],
  auto_class_weights = TRUE,
  max_iterations = 20
) AS
SELECT
  species,
  island,
  culmen_length_mm,
  culmen_depth_mm,
  flipper_length_mm,
  body_mass_g,
  sex
FROM
  `bigquery-public-data.ml_datasets.penguins`
WHERE
  body_mass_g IS NOT NULL
  AND sex IS NOT NULL
"""

print("Training model with TRANSFORM clause...")
job = client.query(transform_query)
job.result()
print(f"Model created: {DATASET_ID}.penguin_engineered")
print(f"Training time: {job.ended - job.started}")

In [ ]:
# Evaluate the engineered model
eng_eval_query = f"""
SELECT *
FROM ML.EVALUATE(
  MODEL `{DATASET_ID}.penguin_engineered`
)
"""

eng_eval_df = client.query(eng_eval_query).to_dataframe()
print("Engineered Model Evaluation:")
for col in eng_eval_df.columns:
    val = eng_eval_df[col].iloc[0]
    if isinstance(val, float):
        print(f"  {col:30s} {val:.4f}")
    else:
        print(f"  {col:30s} {val}")

In [ ]:
# Predict with the TRANSFORM model — pass raw data, transformations auto-applied
transform_predict_query = f"""
SELECT
  predicted_species,
  species AS actual_species,
  island,
  body_mass_g
FROM ML.PREDICT(
  MODEL `{DATASET_ID}.penguin_engineered`,
  (
    SELECT *
    FROM `bigquery-public-data.ml_datasets.penguins`
    WHERE body_mass_g IS NOT NULL
      AND sex IS NOT NULL
    LIMIT 10
  )
)
"""

transform_pred_df = client.query(transform_predict_query).to_dataframe()
print("Predictions with TRANSFORM model (raw data input):")
transform_pred_df

## 9. Model Comparison: Train Multiple Model Types

Train a boosted tree classifier on the same data and compare metrics
with the logistic regression model.

In [ ]:
# Train a boosted tree classifier
boosted_tree_query = f"""
CREATE OR REPLACE MODEL `{DATASET_ID}.penguin_boosted_tree`
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['species'],
  auto_class_weights = TRUE,
  num_parallel_tree = 3,
  max_tree_depth = 5,
  learn_rate = 0.1,
  early_stop = TRUE,
  data_split_method = 'AUTO_SPLIT'
) AS
SELECT
  species,
  island,
  culmen_length_mm,
  culmen_depth_mm,
  flipper_length_mm,
  body_mass_g,
  sex
FROM
  `bigquery-public-data.ml_datasets.penguins`
WHERE
  body_mass_g IS NOT NULL
  AND sex IS NOT NULL
"""

print("Training boosted tree classifier...")
print("(This may take 2-5 minutes)")

job = client.query(boosted_tree_query)
job.result()
print(f"Model created: {DATASET_ID}.penguin_boosted_tree")
print(f"Training time: {job.ended - job.started}")

In [ ]:
# Compare both models
models = [
    ("Logistic Regression", f"{DATASET_ID}.penguin_logistic"),
    ("Boosted Tree", f"{DATASET_ID}.penguin_boosted_tree"),
    ("Logistic + Feature Eng.", f"{DATASET_ID}.penguin_engineered"),
]

comparison_results = []

for name, model_id in models:
    eval_query = f"""
    SELECT *
    FROM ML.EVALUATE(MODEL `{model_id}`)
    """
    try:
        eval_df = client.query(eval_query).to_dataframe()
        row = {"model": name}
        for col in eval_df.columns:
            val = eval_df[col].iloc[0]
            if isinstance(val, float):
                row[col] = round(val, 4)
        comparison_results.append(row)
    except Exception as e:
        print(f"Error evaluating {name}: {e}")

comparison_df = pd.DataFrame(comparison_results)
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
comparison_df

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt

metrics_to_plot = [col for col in comparison_df.columns
                   if col != "model" and comparison_df[col].dtype == float]

if metrics_to_plot:
    fig, axes = plt.subplots(1, min(len(metrics_to_plot), 4), figsize=(16, 5))
    if len(metrics_to_plot) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics_to_plot[:4]):
        bars = ax.bar(comparison_df["model"], comparison_df[metric],
                      color=["#3b82f6", "#22c55e", "#f97316"])
        ax.set_title(metric, fontsize=12, fontweight="bold")
        ax.set_ylim(0, max(comparison_df[metric]) * 1.2)
        ax.tick_params(axis="x", rotation=30)
        for bar, val in zip(bars, comparison_df[metric]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", fontsize=10)

    plt.tight_layout()
    plt.suptitle("Model Comparison: Penguins Classification", y=1.02,
                 fontsize=14, fontweight="bold")
    plt.show()
else:
    print("No numeric metrics to plot.")

## 10. Cleanup (Optional)

Delete the models and dataset created during this tutorial to avoid storage charges.

In [ ]:
# Uncomment to clean up

# # Delete models
# for model_name in ["penguin_logistic", "penguin_boosted_tree",
#                     "penguin_engineered", "sales_arima"]:
#     try:
#         client.delete_model(f"{DATASET_ID}.{model_name}")
#         print(f"Deleted model: {model_name}")
#     except Exception as e:
#         print(f"Could not delete {model_name}: {e}")

# # Delete dataset
# client.delete_dataset(DATASET_ID, delete_contents=True)
# print(f"Deleted dataset: {DATASET_ID}")

print("Uncomment the cleanup code above to delete models and dataset.")

## Summary

In this notebook, we covered:

1. **BigQuery Connection** — Initialize the client and create a dataset
2. **Data Exploration** — Explore the public penguins dataset
3. **Classification (LOGISTIC_REG)** — CREATE MODEL with logistic regression
4. **ML.EVALUATE** — Interpret accuracy, precision, recall, F1, ROC AUC
5. **ML.PREDICT** — Generate batch predictions
6. **Time Series (ARIMA_PLUS)** — Forecast daily sales with holiday effects
7. **ML.EXPLAIN_PREDICT** — Feature importance via Shapley values
8. **TRANSFORM Clause** — Embed feature engineering (BUCKETIZE, FEATURE_CROSS) in the model
9. **Model Comparison** — Train logistic regression vs boosted tree vs engineered, compare metrics

### Key Exam Points
- `CREATE MODEL` with appropriate `model_type` for the task
- Use `TRANSFORM` clause to prevent training-serving skew
- `ML.EVALUATE` returns different metrics for classification vs regression
- `ML.FORECAST` (not ML.PREDICT) for ARIMA_PLUS models
- Export models to Vertex AI for online serving needs